# Phase 3 & 4: Decoupled Recommender System & Dynamic Health-Taste Optimization
## Project: NutriAdapt — Efficient Fine-Tuned LLM for Adaptive Multimodal Nutrition Recommendation
**Reference**: *Personalized Nutrition-Aware Dietary Recommendation With Multimodal LLMs* (IEEE Access, Feb 2026)

### Objectives:
1. **Decouple Recommender from LLM**: Establish fast candidate retrieval and ranking independent of text generation (resolving the paper's latency bottlenecks).
2. **Implement Sequential Recommendation (SASRec)**: Self-Attention Sequential Recommendation in PyTorch with multi-head attention and causal masking.
3. **Implement Baseline Recommenders**: Category Popularity and Content-Based TF-IDF/Nutrition matching.
4. **Adaptive Health-vs-Taste Optimizer**: Implement dynamic Pareto trade-off weighting $\lambda(u, \text{context})$ balancing taste preference against explicit nutritional targets and allergy constraints.
5. **Benchmarking & Candidate Generation**: Compute HR@K, NDCG@K, and export ranked candidate meals for LLM explanation fine-tuning.

In [1]:
# Step 1: Install required dependencies
%pip install -q torch torchvision torchaudio pandas numpy scikit-learn fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 29.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 112.4 MB/s eta 0:00:00


In [2]:
import os
import math
import random
from collections import defaultdict
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Step 2: Configure Paths and Load Preprocessed Data
try:
    from google.colab import drive
    print("Detected Google Colab. Mounting Google Drive...")
    drive.mount('/content/drive')
    colab_candidates = [
        "/content/drive/MyDrive/capstone",
        "/content/drive/MyDrive/CAPSTONE",
        "/content/drive/MyDrive/Capstone"
    ]
    BASE_DIR = next((p for p in colab_candidates if os.path.exists(p)), colab_candidates[0])
except ImportError:
    print("Running in local environment.")
    local_candidates = [
        os.path.abspath("mydrive/capstone"),
        os.path.abspath("mydrive/CAPSTONE"),
        os.path.abspath(".")
    ]
    BASE_DIR = next((p for p in local_candidates if os.path.exists(p)), local_candidates[0])

PROCESSED_DIR = os.path.join(BASE_DIR, "processed")
os.makedirs(PROCESSED_DIR, exist_ok=True)
print(f"Processed Data Directory: {PROCESSED_DIR}")

# Load enriched recipes and splits
recipe_file = os.path.join(PROCESSED_DIR, "recipes_enriched.parquet")
train_file = os.path.join(PROCESSED_DIR, "interactions_train.parquet")
test_file = os.path.join(PROCESSED_DIR, "interactions_test.parquet")

if os.path.exists(recipe_file) and os.path.exists(train_file):
    df_recipes = pd.read_parquet(recipe_file)
    df_train = pd.read_parquet(train_file)
    df_test = pd.read_parquet(test_file)
    print(f"✓ Loaded {len(df_recipes):,} recipes and {len(df_train):,} train interactions!")
else:
    print("Preprocessed files not found. Creating synthetic demonstration data...")
    recipe_ids = list(range(1, 201))
    df_recipes = pd.DataFrame({
        'recipe_id': recipe_ids,
        'recipe_name': [f'Healthy Meal {i}' for i in recipe_ids],
        'ingredients': ['salmon^rice^broccoli' if i%3==0 else ('chicken^quinoa^spinach' if i%3==1 else 'tofu^beans^peppers') for i in recipe_ids],
        'calories': np.random.uniform(250, 650, size=len(recipe_ids)).round(1),
        'protein_g': np.random.uniform(15, 45, size=len(recipe_ids)).round(1),
        'fat_g': np.random.uniform(5, 25, size=len(recipe_ids)).round(1),
        'carbs_g': np.random.uniform(20, 70, size=len(recipe_ids)).round(1),
        'sodium_mg': np.random.uniform(150, 900, size=len(recipe_ids)).round(1),
        'fiber_g': np.random.uniform(2, 12, size=len(recipe_ids)).round(1)
    })
    # Sequential interactions
    users = np.repeat(np.arange(1, 151), 8)
    items = np.random.choice(recipe_ids, size=len(users))
    df_train = pd.DataFrame({'user_id': users[:-150], 'recipe_id': items[:-150]})
    df_test = pd.DataFrame({'user_id': users[-150:], 'recipe_id': items[-150:]})

display(df_recipes.head(3))


Mounted at /content/drive
Processed Data Directory: /content/drive/MyDrive/capstone/processed
✓ Loaded 45,630 recipes and 593,262 train interactions!


,recipe_id,recipe_name,ingredients,cooking_directions,calories,protein_g,fat_g,carbs_g,sodium_mg,fiber_g,nutrition_confidence,nutrition_source
0,240488,"Pork Loin, Apples, and Sauerkraut",sauerkraut drained^Granny Smith apples sliced^...,{'directions': u'Prep\n15 m\nCook\n2 h 30 m\nR...,371.7219,36.39878,11.67521,32.081760,2606.764,10.223360,1.0,source_verified
1,218939,Foolproof Rosemary Chicken Wings,chicken wings^sprigs rosemary^head garlic^oliv...,"{'directions': u""Prep\n20 m\nCook\n40 m\nReady...",335.1655,23.91265,23.62519,6.071931,762.805,0.886681,1.0,source_verified
2,87211,Chicken Pesto Paninis,focaccia bread quartered^prepared basil pesto^...,{'directions': u'Prep\n15 m\nCook\n5 m\nReady ...,640.5617,32.37537,29.36761,60.859430,1075.527,4.441335,1.0,source_verified


In [3]:
# Step 3: Implement Recommender Baselines (Popularity & Content-Based)
print("=== Baseline 1: Popularity Recommender ===")
item_counts = df_train['recipe_id'].value_counts()
pop_recommendations = item_counts.index.tolist()[:20]
print(f"Top-5 Most Popular Recipes: {pop_recommendations[:5]}\n")

print("=== Baseline 2: Content-Based Nutritional Recommender ===")
# Construct combined sensory + nutritional representation
df_recipes['content_text'] = df_recipes['recipe_name'] + ' ' + df_recipes['ingredients'].fillna('')
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_matrix = tfidf.fit_transform(df_recipes['content_text'])

# Normalized nutrition features
nutr_cols = ['calories', 'protein_g', 'fat_g', 'carbs_g', 'sodium_mg', 'fiber_g']
nutr_features = df_recipes[nutr_cols].fillna(0).values
nutr_norm = (nutr_features - nutr_features.mean(axis=0)) / (nutr_features.std(axis=0) + 1e-6)

# Combined feature space
item_content_features = np.hstack([tfidf_matrix.toarray(), nutr_norm])
print(f"Content Item Representation Matrix Shape: {item_content_features.shape}")

=== Baseline 1: Popularity Recommender ===
Top-5 Most Popular Recipes: [17652, 50644, 23600, 56927, 16066]

=== Baseline 2: Content-Based Nutritional Recommender ===
Content Item Representation Matrix Shape: (45630, 1006)


In [4]:
# Step 4: Implement SASRec (Self-Attention Sequential Recommendation in PyTorch)
class SASRec(nn.Module):
    def __init__(self, num_items, hidden_units=64, num_heads=2, num_layers=2, max_seq_len=50, dropout=0.2):
        super(SASRec, self).__init__()
        self.num_items = num_items
        self.hidden_units = hidden_units
        self.max_seq_len = max_seq_len
        
        # Embeddings (0 is padding token)
        self.item_emb = nn.Embedding(num_items + 1, hidden_units, padding_idx=0)
        self.pos_emb = nn.Embedding(max_seq_len, hidden_units)
        self.emb_dropout = nn.Dropout(dropout)
        
        # Transformer Encoder Blocks with Pre-LayerNorm
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_units,
            nhead=num_heads,
            dim_feedforward=hidden_units * 4,
            dropout=dropout,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.layer_norm = nn.LayerNorm(hidden_units)
        
    def forward(self, seqs):
        batch_size, seq_len = seqs.size()
        positions = torch.arange(seq_len, device=seqs.device).unsqueeze(0).expand(batch_size, -1)
        
        # Item + Position embeddings
        x = self.item_emb(seqs) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        
        # Causal mask (prevent lookahead)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=seqs.device)
        padding_mask = (seqs == 0)
        
        out = self.transformer_encoder(x, mask=causal_mask, is_causal=True)
        out = self.layer_norm(out)
        return out[:, -1, :] # Final sequence representation
    
    def score_all_items(self, user_vector):
        # Computes dot product score against all item embeddings
        all_items = self.item_emb.weight[1:] # Exclude padding token
        return torch.matmul(user_vector, all_items.t())

In [5]:
# Step 5: Train SASRec Model on User Sequences
print("Preparing sequential training data...")

# Map recipe_id to contiguous item index (1..M)
unique_items = df_recipes['recipe_id'].unique()
item2idx = {iid: idx + 1 for idx, iid in enumerate(unique_items)}
idx2item = {idx + 1: iid for idx, iid in enumerate(unique_items)}
num_items = len(unique_items)
max_seq_len = 30

# Aggregate sequential histories
user_seqs = defaultdict(list)
for _, row in df_train.iterrows():
    uid = row['user_id']
    iid = row['recipe_id']
    if iid in item2idx:
        user_seqs[uid].append(item2idx[iid])

# Dataset class
class SeqDataset(Dataset):
    def __init__(self, u_seqs, max_len, num_items):
        self.samples = []
        for uid, seq in u_seqs.items():
            if len(seq) >= 2:
                input_seq = seq[:-1][-max_len:]
                target = seq[-1]
                # Pad sequence
                padded = [0] * (max_len - len(input_seq)) + input_seq
                self.samples.append((torch.tensor(padded, dtype=torch.long), target))
                
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        seq, pos_target = self.samples[idx]
        neg_target = random.randint(1, num_items)
        return seq, pos_target, neg_target

dataset = SeqDataset(user_seqs, max_seq_len, num_items)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Initialize SASRec
sasrec = SASRec(num_items=num_items, hidden_units=64, num_heads=2, num_layers=2, max_seq_len=max_seq_len).to(device)
optimizer = torch.optim.AdamW(sasrec.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

print(f"Training SASRec on {len(dataset)} sequence samples for 15 epochs...")
sasrec.train()
for epoch in range(1, 16):
    total_loss = 0.0
    for seqs, pos_targets, neg_targets in loader:
        seqs = seqs.to(device)
        pos_targets = pos_targets.to(device)
        neg_targets = neg_targets.to(device)
        
        user_emb = sasrec(seqs)
        pos_emb = sasrec.item_emb(pos_targets)
        neg_emb = sasrec.item_emb(neg_targets)
        
        pos_logits = (user_emb * pos_emb).sum(dim=-1)
        neg_logits = (user_emb * neg_emb).sum(dim=-1)
        
        loss = criterion(pos_logits, torch.ones_like(pos_logits)) + criterion(neg_logits, torch.zeros_like(neg_logits))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/15 | Loss: {total_loss / len(loader):.4f}")

print("✓ SASRec training complete!")

Preparing sequential training data...


/tmp/ipykernel_13955/2971723845.py:24: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)


Training SASRec on 30548 sequence samples for 15 epochs...
Epoch 01/15 | Loss: 3.4788
Epoch 05/15 | Loss: 1.3366
Epoch 10/15 | Loss: 0.9619
Epoch 15/15 | Loss: 0.7823
✓ SASRec training complete!


In [6]:
# Step 6: Adaptive Health-vs-Taste Dynamic Optimizer with Allergy Safety Constraints
print("=== Implementing Dynamic Contextual Health-Taste Optimizer ===")

def compute_nutritional_suitability(recipe_row, user_goal):
    """
    Computes nutritional alignment score [0, 1] relative to user dietary goal:
    - 'weight_loss': rewards low calories, high fiber, moderate protein
    - 'muscle_gain': rewards high protein, high calories
    - 'heart_healthy': penalizes sodium (>600mg) and saturated fat, rewards fiber
    - 'diabetic_friendly': penalizes high carbs, rewards fiber and protein
    """
    cal = recipe_row['calories']
    pro = recipe_row['protein_g']
    sod = recipe_row['sodium_mg']
    fib = recipe_row['fiber_g']
    carb = recipe_row['carbs_g']
    
    if user_goal == 'weight_loss':
        score = (1.0 - min(cal / 800.0, 1.0)) * 0.5 + min(fib / 10.0, 1.0) * 0.3 + min(pro / 40.0, 1.0) * 0.2
    elif user_goal == 'muscle_gain':
        score = min(pro / 50.0, 1.0) * 0.6 + min(cal / 700.0, 1.0) * 0.4
    elif user_goal == 'heart_healthy':
        sod_penalty = max(0.0, (sod - 400.0) / 1000.0)
        score = max(0.0, 1.0 - sod_penalty) * 0.6 + min(fib / 8.0, 1.0) * 0.4
    elif user_goal == 'diabetic_friendly':
        carb_penalty = max(0.0, (carb - 30.0) / 70.0)
        score = max(0.0, 1.0 - carb_penalty) * 0.6 + min(fib / 8.0, 1.0) * 0.4
    else:
        score = 0.5
    return float(np.clip(score, 0.0, 1.0))

def dynamic_health_weight(user_state, remaining_budget_pct=1.0):
    """
    Calculates dynamic lambda in [0.1, 0.9] based on user goal urgency
    and remaining calorie/sodium budget.
    """
    base_lambda = 0.4
    if user_state.get('goal_priority') == 'high':
        base_lambda += 0.25
    # If remaining daily budget is tight (<30%), boost health weight
    if remaining_budget_pct < 0.3:
        base_lambda += 0.2
    return float(np.clip(base_lambda, 0.1, 0.9))

def check_hard_safety_constraints(ingredients_str, allergies):
    """Strict safety filter: Never allow allergens or restricted ingredients to pass."""
    if not allergies or pd.isna(ingredients_str):
        return True
    ing_lower = str(ingredients_str).lower()
    for allergen in allergies:
        if allergen.lower().strip() in ing_lower:
            return False # Hard safety violation!
    return True

print("✓ Dynamic Health-Taste Optimizer and Hard Safety Constraints Ready!")

=== Implementing Dynamic Contextual Health-Taste Optimizer ===
✓ Dynamic Health-Taste Optimizer and Hard Safety Constraints Ready!


In [7]:
# Step 7: End-to-End Ranking & Generation of Candidate Pairs for LLM Training
print("Generating hybrid recommendations with dynamic optimization...")

sasrec.eval()
recommendation_records = []

test_user_ids = df_test['user_id'].unique()[:50] # Benchmark on 50 test users
user_goal_profiles = ['weight_loss', 'heart_healthy', 'muscle_gain', 'diabetic_friendly']

with torch.no_grad():
    for uid in test_user_ids:
        seq = user_seqs.get(uid, [])
        if not seq:
            continue
        
        input_seq = seq[-max_seq_len:]
        padded = [0] * (max_seq_len - len(input_seq)) + input_seq
        seq_tensor = torch.tensor([padded], dtype=torch.long, device=device)
        
        # 1. Taste scores from SASRec
        u_emb = sasrec(seq_tensor)
        taste_logits = sasrec.score_all_items(u_emb).squeeze(0).cpu().numpy()
        # Min-max normalize taste scores
        taste_scores = (taste_logits - taste_logits.min()) / (taste_logits.max() - taste_logits.min() + 1e-6)
        
        # User context simulation
        user_goal = random.choice(user_goal_profiles)
        user_allergies = ['peanut'] if uid % 4 == 0 else []
        lam = dynamic_health_weight({'goal_priority': 'high'}, remaining_budget_pct=0.25)
        
        # Score candidate recipes
        candidate_scores = []
        for idx, iid in idx2item.items():
            r_row = df_recipes[df_recipes['recipe_id'] == iid]
            if r_row.empty:
                continue
            r_data = r_row.iloc[0]
            
            # Hard allergy constraint
            if not check_hard_safety_constraints(r_data.get('ingredients', ''), user_allergies):
                continue
                
            taste_s = taste_scores[idx - 1]
            health_s = compute_nutritional_suitability(r_data, user_goal)
            
            # Combined Pareto score
            final_score = (1.0 - lam) * taste_s + lam * health_s
            candidate_scores.append((iid, r_data['recipe_name'], final_score, taste_s, health_s, r_data))
            
        # Sort candidates
        candidate_scores.sort(key=lambda x: x[2], reverse=True)
        top_choice = candidate_scores[0]
        
        recommendation_records.append({
            'user_id': uid,
            'user_goal': user_goal,
            'allergies': user_allergies,
            'dynamic_lambda': lam,
            'recommended_recipe_id': top_choice[0],
            'recommended_recipe_name': top_choice[1],
            'final_score': round(float(top_choice[2]), 4),
            'taste_score': round(float(top_choice[3]), 4),
            'health_score': round(float(top_choice[4]), 4),
            'calories': top_choice[5]['calories'],
            'protein_g': top_choice[5]['protein_g'],
            'sodium_mg': top_choice[5]['sodium_mg'],
            'fiber_g': top_choice[5]['fiber_g']
        })

df_recs = pd.DataFrame(recommendation_records)
print(f"\n✓ Generated {len(df_recs):,} personalized, health-optimized meal recommendations!")
display(df_recs.head(5))

# Save recommendation artifacts
rec_save_file = os.path.join(PROCESSED_DIR, "recommender_predictions.parquet")
df_recs.to_parquet(rec_save_file, index=False)
print(f"✓ Saved recommendation candidates to: {rec_save_file}")
print("Ready for Phase 5 & 6 (LLM SFT Fine-Tuning & Hugging Face Push)!")

Generating hybrid recommendations with dynamic optimization...

✓ Generated 50 personalized, health-optimized meal recommendations!


,user_id,user_goal,allergies,dynamic_lambda,recommended_recipe_id,recommended_recipe_name,final_score,taste_score,health_score,calories,protein_g,sodium_mg,fiber_g
0,16,muscle_gain,[peanut],0.85,148498,Beer-Boiled Shrimp,0.9717,0.8111,1.0000,1030.103,78.576470,2166.63300,0.425000
1,32,muscle_gain,[peanut],0.85,148498,Beer-Boiled Shrimp,0.9698,0.7985,1.0000,1030.103,78.576470,2166.63300,0.425000
2,39,weight_loss,[],0.85,8722,Mexican Chicken I,0.7558,0.5158,0.7982,301.237,37.287840,1098.96700,10.748200
3,268,heart_healthy,[peanut],0.85,160708,Strawberry Banana Protein Smoothie,0.9706,0.8042,1.0000,349.420,21.019650,194.53200,10.006000
4,338,heart_healthy,[],0.85,18213,Momma's Potatoes,0.9707,0.8049,1.0000,450.068,8.645125,26.23906,9.720522


✓ Saved recommendation candidates to: /content/drive/MyDrive/capstone/processed/recommender_predictions.parquet
Ready for Phase 5 & 6 (LLM SFT Fine-Tuning & Hugging Face Push)!
